In [ ]:
import pandas as pd
import re
from pathlib import Path


DATA_DIR = Path.home() 

#dataset = pd.read_csv(DATA_DIR / "2023_merged_dataset1.csv")
# dataset = pd.read_csv(DATA_DIR / "2025_merged_dataset.csv")
dataset = pd.read_csv(DATA_DIR / "2025_merged_dataset2.csv")

# flights = pd.read_csv(DATA_DIR / "2023_hourly_airport_demand_2023_CLEAN.csv")
# weather = pd.read_csv(DATA_DIR / "2023_hourly_weather_data_CLEAN.csv")
# load = pd.read_csv(DATA_DIR / "2023_combined_egse_load_data.csv")

flights = pd.read_csv(DATA_DIR / "2025_hourly_total_flights_data_CLEAN.csv")
weather = pd.read_csv(DATA_DIR / "2025_hourly_weather_data_CLEAN2.csv")
#weather = pd.read_csv(DATA_DIR / "2025_hourly_weather_data_CLEAN.csv")
load = pd.read_csv(DATA_DIR / "2025_combined_egse_load_data.csv")

In [ ]:
#code to check/debug/ make sure things are alright with the merged datasets

def format_hour(value):
    if pd.isna(value):
        return None

    if isinstance(value, str):
        parts = value.split(":")
        hour = int(parts[0])
    else:
        hour = int(value)

    return f"{hour:02d}:00"

load["hour"] = load["Hour"].apply(format_hour)
load = load.drop(columns=["Hour"])
load = load.rename(columns={"FlightDate": "date"})
load = load.rename(columns={"Origin": "airport"})
load["airport"] = load["airport"].astype(str)
load["date"] = pd.to_datetime(load["date"])
load_wide = load.pivot_table(
    index=["date", "hour", "airport"],
    columns="eGSE_percent",
    values="Demand_kW",
    aggfunc="mean"
).reset_index()

load_wide = load_wide.rename(columns={
    0.1: "egse_percent_demand_0_1",
    0.25: "egse_percent_demand_0_25",
    0.5: "egse_percent_demand_0_5",
    0.75: "egse_percent_demand_0_75",
    1.0: "egse_percent_demand_1"
})
load_wide["hour"] = load_wide["hour"].astype(str).str.strip()

C:\Users\alecg\AppData\Local\Temp\ipykernel_39520\2617049324.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  load["date"] = pd.to_datetime(load["date"])


In [106]:
print(dataset.shape)
print(dataset.columns)
print(dataset.info())

(438000, 18)
Index(['date', 'airport', 'Departures', 'Arrivals', 'TotalFlights', 'hour',
       'egse_percent_demand_0_1', 'egse_percent_demand_0_25',
       'egse_percent_demand_0_5', 'egse_percent_demand_0_75',
       'egse_percent_demand_1', 'temperature', 'humidity', 'precipitation',
       'wind_direction', 'wind_speed', 'pressure', 'cloud_cover'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 438000 entries, 0 to 437999
Data columns (total 18 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   date                      438000 non-null  str    
 1   airport                   438000 non-null  str    
 2   Departures                438000 non-null  int64  
 3   Arrivals                  438000 non-null  int64  
 4   TotalFlights              438000 non-null  int64  
 5   hour                      438000 non-null  str    
 6   egse_percent_demand_0_1   438000 non-null  float64
 7   egse_percent_dem

In [107]:
missing = dataset.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(dataset)) * 100

print(pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
}).head(20))

                          missing_count  missing_pct
precipitation                     38249     8.732648
pressure                           2518     0.574886
cloud_cover                        1137     0.259589
wind_direction                      176     0.040183
wind_speed                          138     0.031507
humidity                            136     0.031050
temperature                         134     0.030594
date                                  0     0.000000
airport                               0     0.000000
Departures                            0     0.000000
egse_percent_demand_0_75              0     0.000000
egse_percent_demand_0_5               0     0.000000
egse_percent_demand_0_25              0     0.000000
egse_percent_demand_0_1               0     0.000000
hour                                  0     0.000000
TotalFlights                          0     0.000000
Arrivals                              0     0.000000
egse_percent_demand_1                 0     0.

In [108]:
print(dataset["hour"].value_counts().sort_index())

hour
00:00    18250
01:00    18250
02:00    18250
03:00    18250
04:00    18250
05:00    18250
06:00    18250
07:00    18250
08:00    18250
09:00    18250
10:00    18250
11:00    18250
12:00    18250
13:00    18250
14:00    18250
15:00    18250
16:00    18250
17:00    18250
18:00    18250
19:00    18250
20:00    18250
21:00    18250
22:00    18250
23:00    18250
Name: count, dtype: int64


In [109]:
expected_keys = dataset.groupby(["date", "hour", "airport"]).size()

print("Unique time-airport combos:", len(expected_keys))

Unique time-airport combos: 438000


In [110]:
print(dataset.duplicated(subset=["date", "hour", "airport"]).sum())

0


In [111]:
print(dataset["airport"].value_counts().head(20))
print(dataset["airport"].nunique())

airport
ATL    8760
AUS    8760
BNA    8760
BOS    8760
BWI    8760
CLE    8760
CLT    8760
CMH    8760
CVG    8760
DAL    8760
DCA    8760
DEN    8760
DFW    8760
DTW    8760
EWR    8760
FLL    8760
HNL    8760
HOU    8760
IAD    8760
IAH    8760
Name: count, dtype: int64
50


In [112]:
print(dataset[["temperature", "humidity", "wind_speed", "pressure"]].describe())

         temperature       humidity     wind_speed       pressure
count  437866.000000  437864.000000  437862.000000  435482.000000
mean       16.820388      65.155946      13.050774    1016.750471
std        10.167768      20.827538       8.671397       6.203729
min       -28.300000       2.000000       0.000000     978.800000
25%        10.600000      51.000000       7.000000    1013.300000
50%        18.000000      68.000000      11.200000    1016.800000
75%        24.400000      82.000000      18.400000    1020.100000
max        47.800000     100.000000     106.000000    1043.000000


In [113]:
load_cols = [c for c in dataset.columns if "egse" in c or "demand" in c]

print(dataset[load_cols].describe())

       egse_percent_demand_0_1  egse_percent_demand_0_25  \
count            438000.000000             438000.000000   
mean                142.745868                356.872976   
std                 182.002864                455.002714   
min                   0.000000                  0.000000   
25%                  31.800000                 79.500000   
50%                  79.500000                197.700000   
75%                 175.000000                438.600000   
max                1590.900000               3978.400000   

       egse_percent_demand_0_5  egse_percent_demand_0_75  \
count            438000.000000             438000.000000   
mean                713.749079               1070.621448   
std                 910.001707               1365.004934   
min                   0.000000                  0.000000   
25%                 158.000000                237.500000   
50%                 395.500000                592.000000   
75%                 876.100000         

In [114]:
object_cols = dataset.select_dtypes(include=["object"]).columns
print("Object columns left:", object_cols)

Object columns left: Index(['date', 'airport', 'hour'], dtype='str')


C:\Users\alecg\AppData\Local\Temp\ipykernel_39520\3342182154.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = dataset.select_dtypes(include=["object"]).columns


In [86]:
print("Flights columns:", flights.columns.tolist())
print("Weather columns:", weather.columns.tolist())
print("Load columns:", load.columns.tolist())

Flights columns: ['FlightDate', 'Airport', 'Hour', 'Departures', 'Arrivals', 'TotalFlights']
Weather columns: ['airport', 'date', 'time', 'temperature', 'humidity', 'precipitation', 'wind_direction', 'wind_speed', 'pressure', 'cloud_cover', 'weather_condition_code']
Load columns: ['date', 'airport', 'eGSE_percent', 'Demand_kW', 'hour']


In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path.home() 

flights = pd.read_csv(DATA_DIR / "2023_hourly_airport_demand_2023.csv")
weather = pd.read_csv(DATA_DIR / "2023_hourly_weather_data.csv")
load = pd.read_csv(DATA_DIR / "2023_combined_egse_load_data.csv")


#format hour
def format_hour(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        return f"{int(value.split(':')[0]):02d}:00"
    return f"{int(value):02d}:00"


#standardize hour
flights["hour"] = flights["Hour"].apply(format_hour)
flights.drop(columns=["Hour"], inplace=True)

load["hour"] = load["Hour"].apply(format_hour)
load.drop(columns=["Hour"], inplace=True)

weather["hour"] = weather["time"].apply(format_hour)
weather.drop(columns=["time"], inplace=True)


#standardize column names
flights = flights.rename(columns={"FlightDate": "date", "Airport": "airport"})
load = load.rename(columns={"FlightDate": "date", "Origin": "airport"})
# weather already has correct names


# types of data in columns
for df in [flights, load, weather]:
    df["airport"] = df["airport"].astype(str)
    df["date"] = pd.to_datetime(df["date"])

C:\Users\alecg\AppData\Local\Temp\ipykernel_39520\3958152868.py:48: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["date"])


In [ ]:
#check for duplicates
print("Flights duplicates:",
      flights.duplicated(subset=["FlightDate","Hour","Airport"]).sum())

print("Weather duplicates:",
      weather.duplicated(subset=["date","time","airport"]).sum())

print("Load wide duplicates:",
      load_wide.duplicated(subset=["date","hour","airport"]).sum())

Flights duplicates: 0
Weather duplicates: 0
Load wide duplicates: 0


In [ ]:
#make sure no duplicates/see which duplicates
flights.groupby(["FlightDate","Hour","Airport"]).size().sort_values(ascending=False).head(10)

FlightDate  Hour   Airport
2023-01-01  00:00  ATL        1
                   AUS        1
                   BNA        1
                   BOS        1
                   BWI        1
                   CLE        1
                   CLT        1
                   CMH        1
                   CVG        1
                   DAL        1
dtype: int64

In [ ]:
#see column names
flights.columns

Index(['FlightDate', 'Airport', 'Hour', 'Departures', 'Arrivals',
       'TotalFlights'],
      dtype='str')

In [ ]:
#ensure number of rows in datasets make sense

expected = 50 * 365 * 24

print("Expected rows per airport:", 365 * 24)
print("Expected total rows:", expected)
print("Actual rows:", len(dataset))
print("Match:", len(dataset) == expected)

Expected rows per airport: 8760
Expected total rows: 438000
Actual rows: 438000
Match: True
